In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix, classification_report, precision_recall_curve
from sklearn.svm import SVC
import numpy as np

In [3]:
cuisines_df = pd.read_csv("../data//cleaned_cuisines_R.csv")
cuisines_df.head()

,cuisine,almond,angelica,anise,anise_seed,apple,apple_brandy,apricot,armagnac,artemisia,...,whiskey,white_bread,white_wine,whole_grain_wheat_flour,wine,wood,yam,yeast,yogurt,zucchini
0,indian,0.0,0,0.0,0.0,0.0,0,0.0,0,0,...,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
1,indian,1.0,0,0.0,0.0,0.0,0,0.0,0,0,...,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
2,indian,0.0,0,0.0,0.0,0.0,0,0.0,0,0,...,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
3,indian,0.0,0,0.0,0.0,0.0,0,0.0,0,0,...,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
4,indian,0.0,0,0.0,0.0,0.0,0,0.0,0,0,...,0,0.0,0.0,0.0,0.0,0,0.0,0.0,1.0,0.0


In [6]:
np.shape(cuisines_df)

(3995, 381)

In [7]:
cuisines_label_df = cuisines_df['cuisine']
cuisines_label_df.head()

0    indian
1    indian
2    indian
3    indian
4    indian
Name: cuisine, dtype: object

In [ ]:
# Drop the 'cuisine' column to create the feature DataFrame
cuisines_feature_df = cuisines_df.drop(['cuisine'], axis=1)
cuisines_feature_df.head()

,almond,angelica,anise,anise_seed,apple,apple_brandy,apricot,armagnac,artemisia,artichoke,...,whiskey,white_bread,white_wine,whole_grain_wheat_flour,wine,wood,yam,yeast,yogurt,zucchini
0,0.0,0,0.0,0.0,0.0,0,0.0,0,0,0,...,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
1,1.0,0,0.0,0.0,0.0,0,0.0,0,0,0,...,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
2,0.0,0,0.0,0.0,0.0,0,0.0,0,0,0,...,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
3,0.0,0,0.0,0.0,0.0,0,0.0,0,0,0,...,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
4,0.0,0,0.0,0.0,0.0,0,0.0,0,0,0,...,0,0.0,0.0,0.0,0.0,0,0.0,0.0,1.0,0.0


In [9]:
X_train, X_test, y_train, y_test = train_test_split(cuisines_feature_df, cuisines_label_df, test_size=0.3)

In [13]:
# Solver="lbfgs" is an optimization algorithm that is used to find the best parameters for the logistic regression model.
lr = LogisticRegression(solver="lbfgs")
model = lr.fit(X_train, np.ravel(y_train))

accuracy = model.score(X_test, y_test)
print ("Accuracy is {}".format(accuracy))

Accuracy is 0.8423686405337781


In [14]:
print(f'ingredients: {X_test.iloc[50][X_test.iloc[50]!=0].keys()}')
print(f'cuisine: {y_test.iloc[50]}')

ingredients: Index(['beef_broth', 'chicken', 'nut', 'scallion', 'sesame_seed', 'starch'], dtype='object')
cuisine: korean


In [17]:
test= X_test.iloc[50].values.reshape(-1, 1).T
proba = model.predict_proba(test)
classes = model.classes_
resultdf = pd.DataFrame(data=proba, columns=classes)

topPrediction = resultdf.T.sort_values(by=[0], ascending = [False])
topPrediction.head()

c:\Users\jsand\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


,0
chinese,0.786076
korean,0.176122
japanese,0.028569
thai,0.008837
indian,0.000396


In [18]:
y_pred = model.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

     chinese       0.83      0.73      0.78       253
      indian       0.89      0.95      0.92       220
    japanese       0.75      0.86      0.80       222
      korean       0.86      0.81      0.84       253
        thai       0.89      0.87      0.88       251

    accuracy                           0.84      1199
   macro avg       0.84      0.85      0.84      1199
weighted avg       0.84      0.84      0.84      1199



In [20]:
X_train, X_test, y_train, y_test = train_test_split(cuisines_feature_df, cuisines_label_df, test_size=0.3)

In [21]:
C = 10
# Create different classifiers.
classifiers = {
    'Linear SVC': SVC(kernel='linear', C=C, probability=True,random_state=0),
    'KNN classifier': KNeighborsClassifier(C),
    'SVC': SVC(),
    'RFST': RandomForestClassifier(n_estimators=100),
    'ADA': AdaBoostClassifier(n_estimators=100)
}

In [22]:
n_classifiers = len(classifiers)

for index, (name, classifier) in enumerate(classifiers.items()):
    classifier.fit(X_train, np.ravel(y_train))

    y_pred = classifier.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print("Accuracy (train) for %s: %0.1f%% " % (name, accuracy * 100))
    print(classification_report(y_test,y_pred))

Accuracy (train) for Linear SVC: 86.0% 
              precision    recall  f1-score   support

     chinese       0.80      0.83      0.82       254
      indian       0.91      0.91      0.91       252
    japanese       0.85      0.85      0.85       235
      korean       0.86      0.80      0.83       221
        thai       0.88      0.91      0.89       237

    accuracy                           0.86      1199
   macro avg       0.86      0.86      0.86      1199
weighted avg       0.86      0.86      0.86      1199

Accuracy (train) for KNN classifier: 77.2% 
              precision    recall  f1-score   support

     chinese       0.76      0.67      0.71       254
      indian       0.85      0.85      0.85       252
    japanese       0.67      0.82      0.74       235
      korean       0.86      0.67      0.75       221
        thai       0.76      0.85      0.80       237

    accuracy                           0.77      1199
   macro avg       0.78      0.77      0.77    